# AKE Lab 1 · Setup-Check (Aufgabe 0)

Prüft die Umgebung für die Basis-Labs: Python/PyTorch-Version, CPU-Threads, RAM, MNIST-Bezug und die
Laufzeit des Basis-MLP. Alle Zeilen der Zusammenfassung sollten `OK` zeigen; andernfalls das Protokoll
(Ausgabe der letzten Zelle) in die Sprechstunde mitbringen.

**MNIST:** Ordner `data/MNIST` vom Hochschul-Share in dieses Verzeichnis kopieren (oder Umgebungsvariable
`AKE_DATA` auf den Share-Pfad setzen). Fehlt er, versucht torchvision einen Download (12 MB).


In [ ]:
import sys, os, time, platform
from pathlib import Path
checks = {}

def check(name, ok, info=""):
    checks[name] = (bool(ok), info)
    print(f"[{'OK' if ok else 'FEHLER'}] {name}: {info}")

# 1. Python
check("Python >= 3.11", sys.version_info >= (3, 11), f"{platform.python_version()} ({sys.executable})")

# 2. PyTorch & torchvision
try:
    import torch, torchvision
    check("PyTorch importierbar", True, f"torch {torch.__version__}, torchvision {torchvision.__version__}")
    check("CPU-Threads", torch.get_num_threads() >= 2, f"{torch.get_num_threads()} Threads (Referenz: 4)")
    accel = "CUDA" if torch.cuda.is_available() else ("MPS (Apple Silicon)" if getattr(torch.backends, "mps", None)
             and torch.backends.mps.is_available() else "keine (CPU-Lab, voellig ausreichend)")
    print("       Beschleuniger:", accel)
except Exception as e:
    check("PyTorch importierbar", False, repr(e))

# 3. Jupyter-Stack, matplotlib, numpy
for mod in ["numpy", "matplotlib", "jupyter_core", "ipykernel"]:
    try:
        m = __import__(mod); check(f"{mod} importierbar", True, getattr(m, "__version__", ""))
    except Exception as e:
        check(f"{mod} importierbar", False, repr(e))

# 4. RAM und freier Speicher
try:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
except (ValueError, AttributeError, OSError):
    try:
        import psutil; ram_gb = psutil.virtual_memory().total / 2**30
    except Exception:
        ram_gb = float("nan")
check("RAM >= 8 GB", ram_gb >= 7.5 or ram_gb != ram_gb, f"{ram_gb:.1f} GB (nan = nicht ermittelbar)")
import shutil
free_gb = shutil.disk_usage(".").free / 2**30
check("Freier Speicher >= 15 GB", free_gb >= 15, f"{free_gb:.1f} GB frei")

In [ ]:
# 5. MNIST-Bezug
DATA_ROOT = "../data"
raw = Path(DATA_ROOT) / "MNIST" / "raw"
have_files = all((raw / f).exists() for f in [
    "train-images-idx3-ubyte",
    "train-labels-idx1-ubyte",
    "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte",
])
print("MNIST-Rohdaten vorhanden unter", raw.resolve(), ":", have_files,
      "" if have_files else "-> torchvision versucht Download")
try:
    from torchvision import datasets, transforms
    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    train = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=tf)
    test  = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=tf)
    check("MNIST geladen", len(train) == 60_000 and len(test) == 10_000, f"{len(train)} Train / {len(test)} Test")
except Exception as e:
    check("MNIST geladen", False, repr(e))

In [ ]:
# 6. Laufzeit-Referenz: 100 Trainingsschritte des Basis-MLP (784-128-128-10, Adam, Batch 64), hochgerechnet auf eine Epoche
import torch.nn as nn
from torch.utils.data import DataLoader
torch.manual_seed(0)
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 10))
opt, lossf = torch.optim.Adam(model.parameters(), lr=1e-3), nn.CrossEntropyLoss()
dl = DataLoader(train, batch_size=64, shuffle=True)
n_batches = len(dl)
t0 = time.time()
for i, (x, y) in enumerate(dl):
    loss = lossf(model(x), y); opt.zero_grad(); loss.backward(); opt.step()
    if i == 99: break
per_epoch = (time.time() - t0) / 100 * n_batches
check("Laufzeit Basis-MLP <= 45 s/Epoche", per_epoch <= 45,
      f"{per_epoch:.1f} s pro Epoche hochgerechnet ({n_batches} Batches); Lab-Gesamtbudget ~ {per_epoch*22/60:.0f} Min.")
check("Verlust sinkt nach 100 Schritten", loss.item() < 1.5, f"Verlust nach 100 Schritten = {loss.item():.3f} (Start ~2.30)")

In [ ]:
# 7. Zusammenfassung (dieses Protokoll bei Problemen vorzeigen)
print("=" * 72)
print(f"Setup-Check AKE Lab 1 | {platform.platform()} | {time.strftime('%Y-%m-%d %H:%M')}")
print("=" * 72)
alle_ok = True
for name, (ok, info) in checks.items():
    alle_ok &= ok
    print(f"  {'OK    ' if ok else 'FEHLER'}  {name:36s} {info}")
print("=" * 72)
print("ALLES GRUEN -- weiter mit lab01_starter.ipynb" if alle_ok else "MINDESTENS EIN CHECK FEHLGESCHLAGEN -- siehe Aufgabenblatt, Aufgabe 0")